# NB 58 — Subspace steering: English vs multilingual (56 vs 57)

GPU-free comparison of the two subspace-steering experiments, which are identical
in every respect except the lemma language set used to build the concepts:

- **NB 56 — English** (`SupportedLanguages.Default`, `MIN_LEMMAS=4`).
- **NB 57 — multilingual** (`SupportedLanguages.Llama`, 8 languages, `MIN_LEMMAS=11`).

Both run the same recipe (`k=4, steps=24`, bf16 Llama-3.1-8B), the same validated
pool (NB 32, 40 concepts) and FRH pairs (NB 33, 10 pairs), the same two regimes
(**A. singles** and **B. blend pairs**), and the same fluency dial `λ ∈ {0, 0.4}`.
So any difference here is attributable purely to the language set of the lemmas.

Reads the already-computed summaries / JSONLs from `resources/`; no model.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

R = Path("resources")
ARMS = ["pooled", "hierarchical"]
LAMBDAS = [0.0, 0.4]

LANGS = {  # label -> (singles summary, blend summary, jsonl)
    "English": (
        "56_subspace_singles_summary.csv",
        "56_subspace_blend_summary.csv",
        "56_subspace_pooled_vs_hier.jsonl",
    ),
    "Multilingual": (
        "57_subspace_ml_singles_summary.csv",
        "57_subspace_ml_blend_summary.csv",
        "57_subspace_ml.jsonl",
    ),
}

## Summary tables — aligned side by side

In [ ]:
def tagged(csv, lang):
    df = pd.read_csv(R / csv)
    df.insert(0, "lang", lang)
    return df


singles = pd.concat(
    [tagged(LANGS[name][0], name) for name in LANGS], ignore_index=True
).sort_values(["arm", "lam", "lang"])
blend = pd.concat(
    [tagged(LANGS[name][1], name) for name in LANGS], ignore_index=True
).sort_values(["arm", "lam", "lang"])

singles.to_csv(R / "58_lang_comparison_singles.csv", index=False)
blend.to_csv(R / "58_lang_comparison_blend.csv", index=False)

print("A. SINGLES (mean expression Δ)\n")
print(
    singles[["arm", "lam", "lang", "delta", "positive", "ppl_ratio"]]
    .round(3)
    .to_string(index=False)
)
print("\nB. BLEND PAIRS (joint metric)\n")
print(
    blend[["arm", "lam", "lang", "joint_delta", "both_improved", "ppl_ratio"]]
    .round(3)
    .to_string(index=False)
)

## Multilingual − English deltas

Positive = the 8-language build scores higher than English-only for that arm/λ. The headline: multilingual lifts **pooled**, barely moves **hierarchical**.

In [ ]:
def pivot_delta(df, value):
    p = df.pivot_table(index=["arm", "lam"], columns="lang", values=value)
    p["ML - En"] = p["Multilingual"] - p["English"]
    return p.round(3)


print(f"Singles Δ (delta):\n{pivot_delta(singles, 'delta')}\n")
print(f"Blend joint_delta:\n{pivot_delta(blend, 'joint_delta')}\n")
print(f"Blend both_improved:\n{pivot_delta(blend, 'both_improved')}")

## Per-concept singles Δ (λ=0): which concepts gain from more languages?

In [ ]:
def single_deltas(jsonl):
    """Per (arm, concept) single-concept expression Δ at λ=0, averaged over prompts."""
    rows = []
    for line in (R / jsonl).read_text().strip().split("\n"):
        r = json.loads(line)
        c = r["config"]
        if c["mode"] != "single" or c["lam"] != 0.0:
            continue
        name = c["concept"]
        rows.append(
            {"arm": c["arm"], "concept": name, "delta": r["expression"][name]["delta"]}
        )
    # One mean Δ per (arm, concept) so the two languages join on a unique key.
    return pd.DataFrame(rows).groupby(["arm", "concept"], as_index=False)["delta"].mean()


en = single_deltas(LANGS["English"][2]).rename(columns={"delta": "English"})
ml = single_deltas(LANGS["Multilingual"][2]).rename(columns={"delta": "Multilingual"})
per = en.merge(ml, on=["arm", "concept"])
per["ML - En"] = per["Multilingual"] - per["English"]
per.to_csv(R / "58_per_concept_lang_delta.csv", index=False)

print("Mean ML - En by arm (per-concept, λ=0):")
print(per.groupby("arm")["ML - En"].agg(["mean", "median", "size"]).round(3))
print("\nBiggest pooled gainers from multilingual:")
print(
    per[per.arm == "pooled"]
    .nlargest(5, "ML - En")[["concept", "English", "Multilingual", "ML - En"]]
    .round(3)
    .to_string(index=False)
)

## Comparison figure

In [ ]:
lang_colors = {"English": "tab:orange", "Multilingual": "tab:green"}
x = range(len(ARMS))
width = 0.38


def paired_panel(ax, df, value, lam, label):
    for j, lang in enumerate(("English", "Multilingual")):
        vals = []
        for a in ARMS:
            sel = df[(df.arm == a) & (df.lam == lam) & (df.lang == lang)][value]
            vals.append(sel.iloc[0] if len(sel) else float("nan"))
        ax.bar(
            [i + (j - 0.5) * width for i in x],
            vals,
            width,
            label=lang,
            color=lang_colors[lang],
        )
    ax.set_xticks(list(x))
    ax.set_xticklabels(ARMS)
    ax.set_title(f"{label} (λ={lam})")
    ax.axhline(0, color="gray", lw=0.6)


panels = [
    (singles, "delta", "singles Δ"),
    (blend, "joint_delta", "blend joint Δ"),
    (blend, "both_improved", "blend both-improved"),
]
fig, axes = plt.subplots(len(LAMBDAS), len(panels), figsize=(4 * len(panels), 7))
for row, lam in enumerate(LAMBDAS):
    for col, (df, value, label) in enumerate(panels):
        paired_panel(axes[row, col], df, value, lam, label)
axes[0, 0].legend()
plt.suptitle("Subspace steering: English (56) vs multilingual (57)")
plt.tight_layout()
plt.savefig(R / "58_lang_comparison.png", dpi=150)
plt.show()

## Findings

- **Singles.** More languages lift **pooled** (token-vote) but leave
  **hierarchical** (lemma-vote) essentially unchanged — the extra lemmas enrich
  the pooled token cloud, while one-direction-per-lemma does not benefit. This
  widens pooled's single-concept lead in the multilingual regime.
- **Blend.** Multilingual strengthens pooled's composition and softens its
  λ-driven fluency loss; hierarchical's English-only fluency-robustness edge at
  λ=0.4 largely disappears.
- **Net.** The 8-language set favors the **pooled** construction across both
  regimes. Presence-success stays low throughout (expression Δ is the signal),
  and the λ dial roughly halves perplexity in both languages.